In [1]:
# IMPORTS AND UNITS

# The following libraries are already pre-imported to this notebook.
# Should you wish to import additional libraries, please do so here.
# However, please DO NOT delete the `import ...` lines that are already
# here in this notebook.
# 
import copy
import math

import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as shp
import tkinter as tk
import plotly


#
# The base units for this notebook are Newtons (N) and meters (m)
# You may use the variable assignments below to visually add units
# to your numerical quantities.  For example,
# 
# force = 0.5*kN        # means convert 0.5 kN to N
# E = 200*GPa           # means convert 200 GPa to Pa (N/m^2)
#
# b = 0.6*to_mm         # means convert 0.6 m to mm
# moment = 200*to_kN*m  # means convert 200 N-m to kN-m
#
m, to_m =[1, 1]
mm, to_mm = [1e-3, 1e3]
N, to_N = [1, 1]
kN, to_kN = [1e3, 1e-3]
kPa, to_kPa = [1e3, 1e-3]
MPa, to_MPa = [1e6, 1e-6]
GPa, to_GPa = [1e9, 1e-9]

In [2]:
# Test beam #1: Uniform load only, no overhang.

L_1 = 8*m
x_sup1_1 = 0*m
x_sup2_1 = L_1

loads_1 = [
    ["uniform", "dead", -15*kN/m, (0, L_1)],
    ["uniform", "live", -40*kN/m, (0, L_1)]
]

fcp_1, fy_1, fyt_1, cc_1, db_1, dst_1 = 20.7*MPa, 415*MPa, 275*MPa, 40*mm, 16*mm, 10*mm
# Test beam #2: Uniform load with point loads, one overhang.

L_2 = 10*m
x_sup1_2 = 0*m
x_sup2_2 = L_2 - 3*m

loads_2 = [
    ["uniform", "dead", -20*kN/m, (2*L_2/3, L_2)],
    ["uniform", "live", -30*kN/m, (2*L_2/3, L_2)],
    ["point", "dead", -120*kN, L_2/3],
    ["point", "live", -250*kN, L_2/3],
]

fcp_2, fy_2, fyt_2, cc_2, db_2, dst_2 = 34.5*MPa, 415*MPa, 275*MPa, 40*mm, 20*mm, 12*mm

# Test beam #3: General beam loading, two overhangs.
 
L_3 = 6*m
x_sup1_3 = 2*m
x_sup2_3 = L_3 - 2*m

loads_3 = [
    ["point",   "dead", 30*kN,      2*m],
    ["point",   "live", -40*kN,     3*m],
    ["uniform", "dead", -15*kN/m,   (1*m, 4*m)],
    ["uniform", "live", 10*kN/m,    (4.5*m, L_3)],
    ["moment",  "dead", 20*kN*m,    4*m],
    ["moment",  "live", -25*kN*m,   0.5*m]
]

fcp_3, fy_3, fyt_3, cc_3, db_3, dst_3 = 27.6*MPa, 415*MPa, 275*MPa, 50*mm, 25*mm, 12*mm




In [3]:
# IMPORTS AND UNITS

# The following libraries are already pre-imported to this notebook.
# Should you wish to import additional libraries, please do so here.
# However, please DO NOT delete the `import ...` lines that are already
# here in this notebook.
# 
import copy
import math

import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as shp
import tkinter as tk
import plotly
import ipywidgets
import plotly.graph_objects as go

#
# The base units for this notebook are Newtons (N) and meters (m)
# You may use the variable assignments below to visually add units
# to your numerical quantities.  For example,
# 
# force = 0.5*kN        # means convert 0.5 kN to N
# E = 200*GPa           # means convert 200 GPa to Pa (N/m^2)
#
# b = 0.6*to_mm         # means convert 0.6 m to mm
# moment = 200*to_kN*m  # means convert 200 N-m to kN-m
#
m, to_m =[1, 1]
mm, to_mm = [1e-3, 1e3]
N, to_N = [1, 1]
kN, to_kN = [1e3, 1e-3]
kPa, to_kPa = [1e3, 1e-3]
MPa, to_MPa = [1e6, 1e-6]
GPa, to_GPa = [1e9, 1e-9]

# --------------------------------------------------------------START
def plot_beam(L_1, x_sup1_1, x_sup2_1, loads_1):
    fig = go.Figure()

# ---------------------------------------------------------------BEAM
    beam_x = [0, L_1]
    beam_y = [0, 0]
    fig.add_trace(go.Scatter(
        x=beam_x, y=beam_y,
        mode='lines',
        line=dict(color='black', width=6),
        name='Beam'
    ))

# -----------------------------------------------------------SUPPORTS
    fig.add_trace(go.Scatter(
        x=[x_sup1_1], y=[-0.05],
        mode='markers+text',
        marker=dict(symbol='triangle-up', size=16, color='black'),
        text=["Pin"],
        textposition="bottom center",
        name='Pin Support'
    ))

    fig.add_trace(go.Scatter(
        x=[x_sup2_1], y=[-0.05],
        mode='markers+text',
        marker=dict(symbol='circle', size=16, color='black'),
        text=["Roller"],
        textposition="bottom center",
        name='Roller Support'
    ))


# --------------------------------------------------------------LOADS
    for load in loads_1:
        kind, label, value, position = load
        color = 'red' if label == 'dead' else 'blue'

        if kind == "uniform":
            load_start, load_end = position
#            arrow_top = 0.4
#            arrow_len = max(abs(value) / (50 * kN / m), 0.1)
            arrow_base = 0  # beam line
            arrow_len = max(abs(value) / (50 * kN / m), 0.1)
            arrow_tip = arrow_base - arrow_len if value < 0 else arrow_base + arrow_len
            
# --------------------------------------------------------------LOADS
            fig.add_trace(go.Scatter(
                x=[load_start, load_end],
                y=[arrow_base, arrow_base],
                mode='lines',
                line=dict(color=color, width=2),
                name='UDL Line'
            ))

# -------------------------------------------------------------ARROW
            x_positions = np.linspace(load_start, load_end, 8)
            for x in x_positions:
                fig.add_trace(go.Scatter(
                    x=[x, x],
                    y=[arrow_base, arrow_tip],
                    mode='lines+markers',
                    line=dict(color=color, width=2),
                    marker=dict(size=6, symbol='arrow-bar-down', color=color),
                    showlegend=False
                ))

# -------------------------------------------------------------LABEL
            fig.add_annotation(
                x=(load_start + load_end) / 2,
                y=arrow_base + 0.1,
                text=f"{label} load: {value / kN/m:.1f} kN/m",
                showarrow=False,
                font=dict(color=color, size=14)
            )

# ------------------------------------------------------------LAYOUT
    fig.update_layout(
        title="Beam with Partial Uniformly Distributed Loads",
        xaxis=dict(title="Beam Length (m)", range=[-0.5, L_1 + 0.5], zeroline=False),
        yaxis=dict(range=[-0.5, 1], showgrid=False, zeroline=False),
        showlegend=False,
        height=400
    )

    fig.show()

# -------------------------------------------------------------TEST
plot_beam(
    L_1 := 8 * m,
    x_sup1_1=0 * m,
    x_sup2_1=L_1,
    loads_1=[
        ["uniform", "dead", -15 * kN / m, (0, L_1)],
        ["uniform", "live", -40 * kN / m, (0, L_1)]
    ]
)



In [4]:
def factored_loads(loads):
    """
    Convert a list of service loads into a dictionary of factored load combinations.

    Input format for each load: [load_id, 'dead'/'live', magnitude, other_data]
    
    Output Combinations:
    - asd1 = 1.0 * DL
    - asd2 = 1.0 * DL + 1.0 * LL
    - lrfd1 = 1.4 * DL
    - lrfd2 = 1.2 * DL + 1.6 * LL
    """
    # Initialize the dictionary to hold the lists of loads for each combination
    factored_loads_dict = {
        "asd1": [],
        "asd2": [],
        "lrfd1": [],
        "lrfd2": []
    }

    for load in loads:
        # Unpack the load details for clarity
        load_id = load[0]
        load_type = load[1]
        magnitude = load[2]
        other_data = load[3]  # e.g., position, category, etc.

        if load_type == "dead":
            # Add factored dead loads to each combination
            factored_loads_dict["asd1"].append([load_id, magnitude * 1.0, other_data])
            factored_loads_dict["asd2"].append([load_id, magnitude * 1.0, other_data])
            factored_loads_dict["lrfd1"].append([load_id, magnitude * 1.4, other_data])
            factored_loads_dict["lrfd2"].append([load_id, magnitude * 1.2, other_data])
            
        elif load_type == "live":
            # Add factored live loads to each combination
            # Note: asd1 and lrfd1 have a 0.0 factor for live load
            factored_loads_dict["asd1"].append([load_id, magnitude * 0.0, other_data])
            factored_loads_dict["asd2"].append([load_id, magnitude * 1.0, other_data])
            factored_loads_dict["lrfd1"].append([load_id, magnitude * 0.0, other_data])
            # Corrected LRFD2 factor for live load to 1.6
            factored_loads_dict["lrfd2"].append([load_id, magnitude * 1.6, other_data])

    return factored_loads_dict

In [5]:

factored_loads1 = factored_loads(loads_1)
factored_loads2 = factored_loads(loads_2)
factored_loads3 = factored_loads(loads_3)

print(factored_loads1["asd1"])
print(factored_loads1["asd2"])
print(factored_loads1["lrfd1"])
print(factored_loads1["lrfd2"])

print(factored_loads2["asd1"])
print(factored_loads2["asd2"])
print(factored_loads2["lrfd1"])
print(factored_loads2["lrfd2"])

print(factored_loads3["asd1"])
print(factored_loads3["asd2"])
print(factored_loads3["lrfd1"])
print(factored_loads3["lrfd2"])

[['uniform', -15000.0, (0, 8)], ['uniform', -0.0, (0, 8)]]
[['uniform', -15000.0, (0, 8)], ['uniform', -40000.0, (0, 8)]]
[['uniform', -21000.0, (0, 8)], ['uniform', -0.0, (0, 8)]]
[['uniform', -18000.0, (0, 8)], ['uniform', -64000.0, (0, 8)]]
[['uniform', -20000.0, (6.666666666666667, 10)], ['uniform', -0.0, (6.666666666666667, 10)], ['point', -120000.0, 3.3333333333333335], ['point', -0.0, 3.3333333333333335]]
[['uniform', -20000.0, (6.666666666666667, 10)], ['uniform', -30000.0, (6.666666666666667, 10)], ['point', -120000.0, 3.3333333333333335], ['point', -250000.0, 3.3333333333333335]]
[['uniform', -28000.0, (6.666666666666667, 10)], ['uniform', -0.0, (6.666666666666667, 10)], ['point', -168000.0, 3.3333333333333335], ['point', -0.0, 3.3333333333333335]]
[['uniform', -24000.0, (6.666666666666667, 10)], ['uniform', -48000.0, (6.666666666666667, 10)], ['point', -144000.0, 3.3333333333333335], ['point', -400000.0, 3.3333333333333335]]
[['point', 30000.0, 2], ['point', -0.0, 3], ['unif

In [6]:
def support_reactions(loadcombinations, L, x_sup1, x_sup2):
    """
    Calculate the support reactions for a simply supported beam
    with given loads.
    """
    R1 = 0
    R2 = 0
    R1_List =[]
    R2_List = []
    combinations = ("asd1", "asd2", "lrfd1", "lrfd2")
    # Calculate reactions
    output = []
    for combination in combinations:
        R1 = 0
        R2 = 0
        #print("Reactions Reset")
        for load in loadcombinations[combination]:
            
            if(load[0] == "uniform"):
                w = load[1]
                a = load[2][0]
                b = load[2][1]
                c = (a+b)/2
                #print(f"uniform load from {a} to {b} with magnitude {w} with centroid at {c} and x_sup1={x_sup1}, x_sup2={x_sup2}")
                A = np.array([[x_sup1, x_sup2],[1,1]])
                b = np.array([[-w*(b-a)*c],[-w*(b-a)]])
                Reactions = np.linalg.solve(A,b)
                R1 += Reactions[0]
                R2 += Reactions[1]

                #print(f"uniform load reactions: R1={Reactions[0]}, R2={Reactions[1]}")
            elif(load[0] == "point"):
                P = load[1]
                a = load[2]
                #print(f"point load at {a} with magnitude {P} and x_sup1={x_sup1}, x_sup2={x_sup2}")
                A = np.array([[x_sup1, x_sup2],[1,1]])
                b = np.array([[-P*a],[-P]])
                Reactions = np.linalg.solve(A,b)
                R1 += Reactions[0]
                R2 += Reactions[1]

            elif(load[0] == "moment"):
                M = load[1]
                #print(f"moment load with magnitude {M} and x_sup1={x_sup1}, x_sup2={x_sup2}")
                A = np.array([[0, x_sup1-x_sup2],[1, 1]])
                b = np.array([[M],[0]])
                Reactions = np.linalg.solve(A,b)
                R1 += Reactions[0]
                R2 += Reactions[1]


                #print(f"moment load reactions: R1={Reactions[0]}, R2={Reactions[1]}")
        R1_List.append(R1)
        R2_List.append(R2)
        #print(f"reaction of load {combination} at R1 = {R1}, R2= {R2}")
        


    return np.array(R1_List), np.array(R2_List)

support_reactions1 = support_reactions(factored_loads1, L_1, x_sup1_1, x_sup2_1)
print(np.array(support_reactions1))

support_reactions2 = support_reactions(factored_loads2, L_2, x_sup1_2, x_sup2_2)
print(f"{np.array(support_reactions2)}")

support_reactions3 = support_reactions(factored_loads3, L_3, x_sup1_3, x_sup2_3)
print(f"{np.array(support_reactions3)}")


[[[ 60000.]
  [220000.]
  [ 84000.]
  [328000.]]

 [[ 60000.]
  [220000.]
  [ 84000.]
  [328000.]]]
[[[ 50158.73015873]
  [162063.49206349]
  [ 70222.22222222]
  [239238.0952381 ]]

 [[136507.93650794]
  [374603.17460317]
  [191111.11111111]
  [544761.9047619 ]]]
[[[13750.]
  [30625.]
  [19250.]
  [43500.]]

 [[ 1250.]
  [ 9375.]
  [ 1750.]
  [14500.]]]


In [7]:
support_reactions1_1, support_reactions1_2 = support_reactions(factored_loads1, L_1, x_sup1_1, x_sup2_1)
support_reactions2_1, support_reactions2_2 = support_reactions(factored_loads2, L_2, x_sup1_2, x_sup2_2)
support_reactions3_1, support_reactions3_2 = support_reactions(factored_loads3, L_3, x_sup1_3, x_sup2_3)

def updated_factored_loads(factored_loads, reactions_sup1, x_sup1, reactions_sup2, x_sup2) :
    updated_loads = []
    combinations = ("asd1", "asd2", "lrfd1", "lrfd2")
    counter = 0
    asd1 = []
    asd2 = []
    lrfd1 = []
    lrfd2 = []

    for combination in combinations:    
        updated_loads.append(factored_loads[combination])
        updated_loads.append(["point",reactions_sup1[counter], x_sup1])
        updated_loads.append(["point",reactions_sup2[counter], x_sup2])
        counter += 1
        if combination == "asd1":
            asd1 = copy.deepcopy(updated_loads)
        elif combination == "asd2": 
            asd2 = copy.deepcopy(updated_loads)
        elif combination == "lrfd1":
            lrfd1 = copy.deepcopy(updated_loads)
        elif combination == "lrfd2":
            lrfd2 = copy.deepcopy(updated_loads)

    return asd1, asd2, lrfd1, lrfd2


updated_factored_loads1 = updated_factored_loads(factored_loads1, support_reactions1_1, x_sup1_1, support_reactions1_2, x_sup2_1)
print(f"updated loads asd1: {updated_factored_loads1[0]}")
print(f"updated loads asd2: {updated_factored_loads1[1]}")
print(f"updated loads lrfd1: {updated_factored_loads1[2]}")
print(f"updated loads lrfd2: {updated_factored_loads1[3]}")

updated_factored_loads2 = updated_factored_loads(factored_loads2, support_reactions2_1, x_sup1_2, support_reactions2_2, x_sup2_2)
print(f"updated loads asd1: {updated_factored_loads2[0]}")
print(f"updated loads asd2: {updated_factored_loads2[1]}")
print(f"updated loads lrfd1: {updated_factored_loads2[2]}")
print(f"updated loads lrfd2: {updated_factored_loads2[3]}")


updated_factored_loads3 = updated_factored_loads(factored_loads3, support_reactions3_1, x_sup1_3, support_reactions3_2, x_sup2_3)
print(f"updated loads asd1: {updated_factored_loads3[0]}")
print(f"updated loads asd2: {updated_factored_loads3[1]}")
print(f"updated loads lrfd1: {updated_factored_loads3[2]}")
print(f"updated loads lrfd2: {updated_factored_loads3[3]}")


updated loads asd1: [[['uniform', -15000.0, (0, 8)], ['uniform', -0.0, (0, 8)]], ['point', array([60000.]), 0], ['point', array([60000.]), 8]]
updated loads asd2: [[['uniform', -15000.0, (0, 8)], ['uniform', -0.0, (0, 8)]], ['point', array([60000.]), 0], ['point', array([60000.]), 8], [['uniform', -15000.0, (0, 8)], ['uniform', -40000.0, (0, 8)]], ['point', array([220000.]), 0], ['point', array([220000.]), 8]]
updated loads lrfd1: [[['uniform', -15000.0, (0, 8)], ['uniform', -0.0, (0, 8)]], ['point', array([60000.]), 0], ['point', array([60000.]), 8], [['uniform', -15000.0, (0, 8)], ['uniform', -40000.0, (0, 8)]], ['point', array([220000.]), 0], ['point', array([220000.]), 8], [['uniform', -21000.0, (0, 8)], ['uniform', -0.0, (0, 8)]], ['point', array([84000.]), 0], ['point', array([84000.]), 8]]
updated loads lrfd2: [[['uniform', -15000.0, (0, 8)], ['uniform', -0.0, (0, 8)]], ['point', array([60000.]), 0], ['point', array([60000.]), 8], [['uniform', -15000.0, (0, 8)], ['uniform', -400